# COMPRO Final Exam Review: Python Exceptions & Error Types
**Goal:** Learn to recognize errors, predict what happens, design `try`/`except` correctly, and solve exam-style debugging problems.

**How to use this notebook**
1. Read the concept and predict the result before running code.
2. For each question, write your answer in the Markdown space or in a new cell.
3. Run the code only after committing to a prediction.
4. Try the challenge section before opening the answer key.

This notebook uses standard Python behavior and expands the exception-handling ideas for exam practice. If your lecture uses particular names or conventions, prioritize your instructor's wording.

## 1. The big picture: errors are not all the same

A useful exam classification:

| Category | What it means | Typical example | Can `try/except` catch it? |
|---|---|---|---|
| **Syntax error** | Python cannot parse the code | Missing `:` | Not by a `try` around that same invalid syntax; the program fails before running |
| **Runtime exception** | Code starts running, then an operation raises an exception | `10 / 0` | Yes, if the matching `try` covers the operation |
| **Logical error** | Code runs but gives the wrong answer | Using `+` when you meant `*` | Usually not; fix the logic |
| **Input/data error** | The input is invalid for the program's expectations | `int("hello")` | Often raises a runtime exception such as `ValueError` |

Important distinction: **“runtime error” is a broad informal label**. In Python, most runtime failures are represented by exception objects, such as `ZeroDivisionError` or `TypeError`. A logical error may produce no exception at all.


## 2. Recognize common exception types

### Quick reference

- `SyntaxError`: invalid Python grammar; e.g. `if x > 2 print(x)`.
- `NameError`: a name is used before it has been defined.
- `TypeError`: an operation/function receives an inappropriate type or number of arguments. #**"a"+2**
- `ValueError`: the type is acceptable, but the value/format is unsuitable.  #**int("a")**
- `ZeroDivisionError`: division or modulo by zero.
- `IndexError`: a sequence index is out of range.
- `KeyError`: a dictionary key is missing.
- `AttributeError`: an object does not have the requested attribute/method.
- `FileNotFoundError`: a file path does not exist when opening for reading.
- `ImportError` / `ModuleNotFoundError`: an import cannot be completed / module cannot be found.
- `EOFError`: `input()` reaches end-of-file without reading a line.  #ctrl+D for input
- `AssertionError`: an `assert` condition is false.
- `OverflowError`: a numeric operation exceeds a representable range in operations that raise it.
- `StopIteration`: an iterator has no more items; commonly handled internally by loops.
- `Exception`: common base class for most ordinary, catchable application exceptions.

Do not classify an error from one keyword alone. Ask: **what operation failed, and why?**


In [3]:
# Predict each exception type before running:
examples = [
    lambda: missing_name + 1,
    lambda: "3" + 4,
    lambda: int("three"),
    lambda: [10, 20][5],
    lambda: {"name": "Ada"}["age"],
    lambda: 7 / 0,
    lambda: (10).append(2),
]

for i, operation in enumerate(examples, start=1):
    try:
        print(i, operation())
    except Exception as e:
        print(i, type(e).__name__, "-", e)


1 NameError - name 'missing_name' is not defined
2 TypeError - can only concatenate str (not "int") to str
3 ValueError - invalid literal for int() with base 10: 'three'
4 IndexError - list index out of range
5 KeyError - 'age'
6 ZeroDivisionError - division by zero
7 AttributeError - 'int' object has no attribute 'append'


### Check your understanding
For each case, name the most specific exception you expect.

1. `float("3.2.1")` ValueError
2. `[4, 5][0]` is valid, but `[4, 5][2]` is not. ## bc.this is indexing `[4, 5][0]`=>`4` or `[4, 5][1]`=> 5  ,, so `[4, 5][2]` = IndexError
3. `"score" - 1` TypeError
4. `student["id"]` when `"id"` is absent. KeyError
5. `print(total)` when `total` was never assigned. NameError
6. `open("missing.txt")` when the file does not exist. FileNotFoundError 

## 3. Syntax errors vs runtime exceptions vs logical errors

### A. Syntax error
This is invalid grammar, so Python cannot execute the file/cell.

```python
# Invalid: missing colon
if score >= 50
    print("pass")
```

Correct it to `if score >= 50:`.

### B. Runtime exception
The syntax is valid, but an operation fails while running.

```python
score = int("fifty")  # ValueError
```

### C. Logical error
The program runs, but the result is incorrect.

```python
prices = [10, 20, 30]
average = sum(prices) / 2  # No exception, but wrong denominator
```

The correct denominator is `len(prices)`.


## 4. `try`, `except`, `else`, and `finally`

```python
try:
    # Code that might raise an exception
    value = int(input("Number: "))
except ValueError:
    # Runs only if a ValueError is raised in the try suite
    print("Please enter an integer.")
else:
    # Runs if the try suite completes without an exception
    print("You entered", value)
finally:
    # Runs whether an exception happened or not
    print("Finished attempting input.")
```

Remember:
- `except` handles matching exceptions raised in the `try` suite.
- `else` runs only if the `try` suite finishes normally.
- `finally` is for cleanup that should happen regardless (for example, releasing a resource).
- If an exception is not handled, it continues propagating after `finally` runs.
- An exception raised inside an `except` block is not handled by another `except` belonging to that same `try`.


In [ ]:
def classify_number(text):
    try:
        number = int(text)
    except Exception as e:
        print(f"{type(e).__name__} invalid integer {e}") ###
    else:
        print("valid integer:", number)
    finally:
        print("classification finished")

classify_number("12")
print("---")
classify_number("twelve")


valid integer: 12
classification finished
---
ValueError invalid integer invalid literal for int() with base 10: 'twelve'
classification finished


### Exam question 1 — predict the exact output

```python
def demo(text):
    try:
        n = int(text)
        result = 10 // n
    except ValueError:
        print("A")
    except ZeroDivisionError:
        print("B")
    else:
        print("C", result)
    finally:
        print("D")

demo("2")
demo("zero")
demo("0")
```

Write the three output groups in order before running the cell below.

In [11]:
def demo(text):
    try:
        n = int(text)
        result = 10 // n
    except ValueError:
        print("A")
    except ZeroDivisionError:
        print("B")
    else:
        print("C", result)
    finally:
        print("D")
# Run only after you have predicted the output.
demo("2")
demo("zero")
demo("0")


C 5
D
A
D
B
D


## 5. Order and specificity of `except` blocks

Put specific exceptions before broad exceptions.

```python
try:
    value = int(user_text)
except ValueError:
    print("Input was not an integer.")
except Exception:
    print("Some other ordinary exception occurred.")
```

This order is bad:

```python
try:
    value = int(user_text)
except Exception:
    print("Something happened.")
except ValueError:
    print("Invalid integer.")
```

The second handler is unreachable for `ValueError`, because `ValueError` is a subclass of `Exception`. Catching `Exception` everywhere can hide bugs and make debugging harder.

Avoid catching `BaseException` for ordinary program errors; it also includes control-flow exceptions such as `KeyboardInterrupt` and `SystemExit`.


### Exam question 2 — which handler runs?

```python
try:
    int("x")
except TypeError:
    print("type")
except ValueError:
    print("value")
except Exception:
    print("general")
```

What prints, and why? What happens if you swap the `ValueError` and `Exception` handlers?


## 6. Similar-looking errors: `TypeError` vs `ValueError`

- `TypeError`: wrong kind of object / unsupported operation.
- `ValueError`: the kind of object is acceptable, but the value is invalid.

```python
int("abc")       # ValueError: string is a valid input type, but not a valid integer representation
int([1, 2])      # TypeError: list is not an acceptable input type for int()
"3" + 4          # TypeError: cannot add str and int
```

A helpful diagnostic question: **Is the type wrong, or is the value wrong?**


### Exam question 3 — choose the exception

For each expression, write `TypeError`, `ValueError`, or another specific exception.

1. `int("4.5")`  # `ValueError` bc its a str
2. `int(4.5)`  no error
3. `len(10)`  # `TypeError: object of type 'int' has no len()`
4. `10 % 0`  #` ZeroDivisionError`
5. `{}.pop("missing")`  # KeyError: 'missing' bc its a dict
6. `"abc".index("z")`  # ValueError: substring not found
7. `list(5)`  # `TypeError: 'int' object is not iterable`


## 7. Raising exceptions intentionally

Use `raise` when a value violates a rule your program requires.

```python
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError("age cannot be negative")
    return age
```

The distinction is intentional:
- `TypeError` when the caller passes the wrong kind of object.
- `ValueError` when the type is suitable but the value is outside allowed rules.

Do not use `assert` as the main way to validate external input. Assertions are primarily for internal assumptions and can be disabled with optimized Python execution.


In [12]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError("age cannot be negative")
    return age

for item in [18, -1, "18"]:
    try:
        print("result:", set_age(item))
    except (TypeError, ValueError) as error:
        print(type(error).__name__, error)


result: 18
ValueError age cannot be negative
TypeError age must be an integer


### Exam question 4 — design validation

Write `withdraw(balance, amount)` with these rules:
- `balance` and `amount` must be numeric values (you may choose to support only `int` and `float`).
- `amount` must be greater than zero.
- `amount` cannot exceed `balance`.
- Return the new balance if successful.
- Raise a suitable exception for each invalid case.

Think carefully about the order of checks and whether `bool` should count as an integer in Python.


## 8. Custom exceptions

A custom exception communicates a domain-specific problem.

```python
class InvalidScoreError(ValueError):
    pass

def validate_score(score):
    if not 0 <= score <= 100:
        raise InvalidScoreError("score must be between 0 and 100")
```

Subclassing `ValueError` means callers can catch either the specific error or the broader `ValueError`.

```python
try:
    validate_score(130)
except InvalidScoreError as e:
    print("Score problem:", e)
```

Exam design principle: create a custom exception when it makes the program's error meaning clearer—not merely to rename every built-in exception.


## 9. Exceptions and files

Use `with open(...)` to manage file closing reliably.

```python
try:
    with open("scores.txt", "r", encoding="utf-8") as file:
        contents = file.read()
except FileNotFoundError:
    print("The file does not exist.")
except PermissionError:
    print("Permission denied.")
```

A common exam pattern: a line exists, but its content is malformed. Opening can succeed, while conversion fails.

```python
def parse_score(line):
    try:
        return int(line.strip())
    except ValueError:
        return None
```

Consider whether `None` is an appropriate fallback. In a real application, silently discarding bad data may hide important problems.


## 10. `return`, `finally`, and propagation — advanced behavior

When an exception is raised, Python searches outward for a matching handler. If none is found, the exception propagates and the program reports a traceback.

Be cautious with `return` in `finally`: it can override a return value from `try` and can suppress an active exception. Usually, avoid returning from `finally`.

```python
def risky():
    try:
        return 1
    finally:
        print("cleanup")
```

This returns `1` after printing `cleanup`.

Now reason about this less-recommended example:

```python
def confusing():
    try:
        return 1
    finally:
        return 2
```

The result is `2`, because the `finally` return overrides the earlier return.


## 11. Exam-style output tracing

For each problem:
1. Identify the first line that may fail.
2. Name the exception type.
3. Decide which handler matches first.
4. Check whether `else` runs.
5. Check whether `finally` runs.
6. Determine whether execution continues after the `try` statement.


### Problem A — list and dictionary access

Predict the output exactly.

```python
data = {"nums": [3, 6, 9]}

try:
    print(data["nums"][3])
    print("after index")
except KeyError:
    print("key problem")
except IndexError:
    print("index problem")
finally:
    print("done")

print("program continues")
```


### Problem B — exception in an `except` block

Predict the output and explain why the `TypeError` is not caught by the second handler.

```python
try:
    int("oops")
except ValueError:
    print("bad value")
    print(5 + "5")
except TypeError:
    print("bad type")
finally:
    print("always")
```


### Problem C — nested `try`

```python
try:
    try:
        1 / 0
    except ValueError:
        print("inner value")
    print("after inner")
except ZeroDivisionError:
    print("outer zero")
finally:
    print("finish")
```

Which lines print? Explain where the exception travels.


## 12. Debugging challenge: fix the function

The function should return an integer parsed from text, and return `None` if the text is invalid. However, it has several problems.

```python
def parse_int(text):
    try:
        number = int(text)
    except Exception:
        print("bad input")
    else:
        return number
    finally:
        return None
```

Tasks:
1. Explain why valid input still returns `None`.
2. Explain why catching `Exception` is broader than necessary.
3. Rewrite the function so valid text returns an integer and invalid text returns `None`.
4. Should a file error be swallowed by this function? Why or why not?


## 13. Hard integrated challenge — parse a score file

Each line of a file is expected to contain `student_name,score`. Example:

```text
Ari,88
Mina,not-a-number
,72
Bo,105
Chen,91
```

Write `parse_records(lines)` that:
- returns a list of `(name, score)` tuples for valid records;
- requires a non-empty name;
- requires an integer score from 0 through 100;
- skips invalid lines but records a useful reason for each rejected line;
- does not stop processing just because one line is invalid.

**Challenge rules:** Use specific exceptions and `raise` for validation. Do not use `except Exception:` as a blanket solution. Decide whether your own `InvalidRecordError` should inherit from `ValueError`.


## 14. Mini mock exam (no running until you finish)

**Q1.** Which category best describes `if x > 2 print(x)`?
A. `ValueError`  B. `SyntaxError`  C. `NameError`  D. logical error

**Q2.** `int("ten")` raises which exception?

**Q3.** `values = [4, 8]`; what happens with `values[2]`?

**Q4.** Which block runs when the `try` suite completes without raising an exception: `except`, `else`, or `finally`?

**Q5.** Does `finally` normally run when the `try` suite raises an unhandled exception?

**Q6.** Why should `except ValueError` usually appear before `except Exception`?

**Q7.** What is the difference between `raise` and `return`?

**Q8.** Is `average = sum(scores) / 2` necessarily an exception if there are three scores? Explain.

**Q9.** What exception is raised by `{"x": 1}["y"]`?

**Q10.** Why is `except:` often a poor choice for ordinary application error handling?

**Q11.** Write a safe conversion function that returns `None` for invalid integer text.

**Q12.** In a `try/except/else/finally` statement, does an exception raised inside the `else` suite get handled by the `except` clauses belonging to that same `try`?


## 15. Answer key and explanations

### Common exception types
1. `float("3.2.1")` → `ValueError`
2. `[4, 5][2]` → `IndexError`
3. `"score" - 1` → `TypeError`
4. Missing dictionary key → `KeyError`
5. Undefined variable → `NameError`
6. Missing file opened for reading → `FileNotFoundError`

### Exam question 1
`demo("2")` prints:
```text
C 5
D
```
`demo("zero")` prints:
```text
A
D
```
`demo("0")` prints:
```text
B
D
```
`else` runs only when the `try` suite has no exception; `finally` runs in all three cases.

### Exam question 2
Prints `value`, because `int("x")` raises `ValueError`. If `except Exception` is placed before `except ValueError`, the broad handler catches it first, so `value` is never printed.

### Exam question 3
1. `int("4.5")` → `ValueError`
2. `int(4.5)` → valid conversion to `4` (truncates toward zero)
3. `len(10)` → `TypeError`
4. `10 % 0` → `ZeroDivisionError`
5. `{}.pop("missing")` → `KeyError`
6. `"abc".index("z")` → `ValueError`
7. `list(5)` → `TypeError`

### Problem A
Prints:
```text
index problem
done
program continues
```
The first `print` attempts an out-of-range list index and raises `IndexError`. The remaining statement in the `try` suite is skipped. The matching handler runs, then `finally`, then execution continues.

### Problem B
Prints:
```text
bad value
always
```
Then the `TypeError` propagates with a traceback. The `except TypeError` in the same `try` does not catch exceptions raised inside the `except ValueError` suite. `finally` still runs.

### Problem C
Prints:
```text
outer zero
finish
```
The inner `except ValueError` does not match `ZeroDivisionError`, so it propagates to the outer `try`. The `print("after inner")` is skipped.

### Debugging challenge — one correct version
```python
def parse_int(text):
    try:
        return int(text)
    except ValueError:
        print("bad input")
        return None
```
No `finally` is needed. `TypeError` may still propagate if the caller passes a non-string object that `int()` cannot convert; whether to handle it depends on the function's contract. Do not hide unrelated failures by catching every `Exception`.

### Mini mock exam
1. B — `SyntaxError`.
2. `ValueError`.
3. `IndexError`.
4. `else`.
5. Yes, under normal control flow, `finally` runs even if the exception is unhandled; then the exception continues propagating.
6. A broad handler first would catch the specific exception, making the later specific handler unreachable for that exception.
7. `return` sends a result back to the caller; `raise` signals an exceptional condition and transfers control to exception handling.
8. No. It is a logical error if the intended average is over all three values, but Python may execute it without raising an exception.
9. `KeyError`.
10. A bare `except:` catches almost everything, including control-flow exceptions that usually should not be swallowed.
11.
```python
def safe_int(text):
    try:
        return int(text)
    except ValueError:
        return None
```
12. No. An exception raised in the `else` suite is not handled by the `except` clauses associated with that same `try`.


## 16. Final checklist

You are exam-ready when you can do all of these without notes:

- [ ] Distinguish syntax, runtime, and logical errors.
- [ ] Identify `TypeError` vs `ValueError`.
- [ ] Recognize `NameError`, `IndexError`, `KeyError`, `AttributeError`, `ZeroDivisionError`, and `FileNotFoundError`.
- [ ] Trace `try/except/else/finally` output.
- [ ] Explain exception matching and why handler order matters.
- [ ] Use `raise` to validate input.
- [ ] Define and catch a custom exception.
- [ ] Explain exception propagation and nested `try` behavior.
- [ ] Avoid using `finally` to return a value.
- [ ] Debug code that accidentally hides exceptions.
- [ ] Process many records while rejecting bad records individually.

**Study strategy:** first attempt every output-tracing question without executing it. Then run it, compare, and explain every difference in your own words.
